In [ ]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import numpy.typing as npt
import polars as pl
import polars.selectors as cs

In [ ]:
@dataclass
class ModelData:
    covariates: npt.NDArray[np.float64]
    beliefs: npt.NDArray[np.int64]
    covariate_names: list[str]
    belief_names: list[str]


def prepare_model_data(data: pl.DataFrame) -> ModelData:
    n_waves = data.select(pl.col("survey_wave").unique()).shape[0]
    covariate_names = [
        col[len("covariate_") :]
        for col in data.select(cs.starts_with("covariate_")).columns
    ]
    belief_names = [
        col[len("belief_") :] for col in data.select(cs.starts_with("belief_")).columns
    ]

    covariates = (
        data.select(cs.starts_with("covariate_"))
        .to_numpy()
        .ravel()  # Unwrap into 1D array
        .reshape(
            (-1, n_waves, len(covariate_names))
        )  # Reshape into (participant, wave, covariate)
    )

    beliefs = (
        data.select(cs.starts_with("belief_"))
        .to_numpy()
        .ravel()  # Unwrap into 1D array
        .reshape(
            (-1, n_waves, len(belief_names))
        )  # Reshape into (participant, wave, covariate)
    )

    return ModelData(
        covariates=covariates,
        beliefs=beliefs,
        covariate_names=covariate_names,
        belief_names=belief_names,
    )

In [ ]:
model_data_dir = Path(
    "/Users/henry/data/msc_thesis/climate-attitudes/w3w4_dataset/model_data.parquet"
)
model_data_imputed_dir = Path(
    "/Users/henry/data/msc_thesis/climate-attitudes/w3w4_dataset/model_data_imputed.parquet"
)

In [ ]:
model_data = prepare_model_data(pl.read_parquet(model_data_dir))
model_data_imputed = prepare_model_data(pl.read_parquet(model_data_imputed_dir))

In [ ]:
model_data